In [11]:
import numpy as np
import pandas as pd

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import f1_score, accuracy_score

# Preprocessing
from sklearn.preprocessing import StandardScaler

# to split data
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('heart.csv')

In [58]:
df.shape

(1025, 14)

In [59]:
cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

In [17]:
scaler = StandardScaler()

In [60]:
X = df.drop(['target'], axis = 1)
y = df['target']

In [61]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.33,
    random_state=42,
    stratify=y
)
scaler.fit(X_train[cols])
X_train[cols] = scaler.transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

In [62]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": GaussianNB(),
    "KNN" : KNeighborsClassifier(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier()
}

In [37]:
results = []

In [38]:
for name, model in models.items() : 
    model.fit(X_train, y_train) 
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred),4),
        "F-1 score": round(f1_score(y_test, y_pred),4)
    })

In [39]:
results

[{'Model': 'Logistic Regression', 'Accuracy': 0.8201, 'F-1 score': 0.8338},
 {'Model': 'Naive Bayes', 'Accuracy': 0.8378, 'F-1 score': 0.8451},
 {'Model': 'KNN', 'Accuracy': 0.8614, 'F-1 score': 0.8653},
 {'Model': 'SVM', 'Accuracy': 0.8791, 'F-1 score': 0.8864},
 {'Model': 'Decision Tree', 'Accuracy': 0.9823, 'F-1 score': 0.9825}]

In [51]:
# K-fold cross validation
from sklearn.model_selection import cross_val_score

In [63]:
cross_results = []

In [70]:
for name, model in models.items() : 
    cross_results.append({
        "Model": name,
        "Accuracy": cross_val_score(model, X, y, cv = 5, scoring = "accuracy")
    })

In [71]:
cross_results

[{'Model': 'Logistic Regression',
  'Accuracy': array([0.88780488, 0.85365854, 0.87804878, 0.82439024, 0.80487805])},
 {'Model': 'Naive Bayes',
  'Accuracy': array([0.87317073, 0.82926829, 0.84390244, 0.7902439 , 0.77073171])},
 {'Model': 'KNN',
  'Accuracy': array([0.87804878, 0.8195122 , 0.81463415, 0.85365854, 0.82926829])},
 {'Model': 'SVM',
  'Accuracy': array([0.93658537, 0.91707317, 0.90731707, 0.84878049, 0.84878049])},
 {'Model': 'Decision Tree',
  'Accuracy': array([0.9804878 , 0.98536585, 1.        , 1.        , 1.        ])},
 {'Model': 'Logistic Regression',
  'Accuracy': array([0.88780488, 0.85365854, 0.87804878, 0.82439024, 0.80487805])},
 {'Model': 'Naive Bayes',
  'Accuracy': array([0.87317073, 0.82926829, 0.84390244, 0.7902439 , 0.77073171])},
 {'Model': 'KNN',
  'Accuracy': array([0.87804878, 0.8195122 , 0.81463415, 0.85365854, 0.82926829])},
 {'Model': 'SVM',
  'Accuracy': array([0.93658537, 0.91707317, 0.90731707, 0.84878049, 0.84878049])},
 {'Model': 'Decision Tre

In [74]:
from sklearn.metrics import confusion_matrix, classification_report

In [75]:
model = models['Decision Tree']
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[165   0]
 [  6 168]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       165
           1       1.00      0.97      0.98       174

    accuracy                           0.98       339
   macro avg       0.98      0.98      0.98       339
weighted avg       0.98      0.98      0.98       339



In [76]:
from sklearn.model_selection import GridSearchCV

In [77]:
classifier = GridSearchCV(models["SVM"], {
    'C' : [1,10,20,30],
    'kernel' : ['linear', 'rbf']
}, cv = 5, return_train_score = False)

In [78]:
classifier.fit(X,y)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [1, 10, 20, 30], 'kernel': ['linear', 'rbf']})

In [81]:
cl_results = pd.DataFrame(classifier.cv_results_)

In [ ]:
print(cl_results)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.027913,0.006530,0.004392,0.000711,1,linear,"{'C': 1, 'kernel': 'linear'}",0.882927,0.848780,0.853659,0.809756,0.804878,0.840000,0.029171,5
1,0.019410,0.000926,0.013685,0.001082,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.936585,0.917073,0.907317,0.848780,0.848780,0.891707,0.036295,4
2,0.138113,0.054792,0.004465,0.000666,10,linear,"{'C': 10, 'kernel': 'linear'}",0.882927,0.848780,0.839024,0.809756,0.804878,0.837073,0.028377,6
3,0.020026,0.000460,0.010044,0.000416,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.951220,0.951220,0.926829,0.951220,0.960976,0.948293,0.011377,3
4,0.241759,0.104242,0.004230,0.000683,20,linear,"{'C': 20, 'kernel': 'linear'}",0.882927,0.848780,0.839024,0.809756,0.804878,0.837073,0.028377,6
5,0.020700,0.001549,0.009554,0.000746,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.960976,0.951220,0.951220,0.965854,0.970732,0.960000,0.007805,2
6,0.292587,0.036108,0.004172,0.001233,30,linear,"{'C': 30, 'kernel': 'linear'}",0.882927,0.848780,0.839024,0.809756,0.804878,0.837073,0.028377,6
7,0.027569,0.008464,0.010340,0.002285,30,rbf,"{'C': 30, 'kernel': 'rbf'}",0.960976,0.960976,0.975610,0.965854,0.975610,0.967805,0.006617,1
